# Chapter 8
## Quadratic Integrate-and-Fire (QIF) and Theta Neurons
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter08.ipynb)

## About this chapter

The quadratic integrate-and-fire (QIF) neuron replaces the LIF model's
linear decay with a quadratic one, giving it a saddle-node bifurcation and
an analytically solvable blow-up to threshold. The theta neuron is an exact
change of variables of the QIF model onto a circle, turning voltage
blow-up/reset into smooth rotation.

The QIF subthreshold/blow-up equation is

$$
\tau_m\frac{dV}{dt}=V(1-V)+\tau_m I,
$$

which for $I>1/(4\tau_m)$ has no fixed point and $V$ blows up to $+\infty$
before resetting from $-\infty$, giving a finite period. The theta-neuron
change of variables $V=\tfrac12(1-\cos\theta)/(\ldots)$ maps this onto

$$
\tau_m\frac{d\theta}{dt}=-\cos\theta+2\tau_m I(1+\cos\theta).
$$

Here $V$ is voltage, $\theta$ is the phase on the circle, $\tau_m$ is the
membrane time constant, and $I$ is a constant drive.

See [`README.md`](chapter08.md) for the full guide, including suggested
order and related chapters.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact
from mnd.core import draw_arrow

## The QIF Neuron's Closed-Form Solution

Above threshold ($I>1/(4\tau_m)$), the QIF equation is exactly solvable;
this reproduces the periodic blow-up/reset trajectory analytically rather
than by integration.

In [ ]:
def simulate_qif_infinite_threshold(tau_m=2.0, i=0.15):
    w = np.sqrt(tau_m * i - 1.0 / 4.0)
    T = 2.0 * tau_m / w * np.arctan(1.0 / (2.0 * w))
    t_ast = tau_m / w * (np.pi / 2.0 - np.arctan(1.0 / (2.0 * w)))

    t_period = np.arange(101) / 100.0 * T
    v_0_to_1 = 0.5 + w * np.tan(w / tau_m * t_period - np.arctan(1.0 / (2.0 * w)))

    t_blowup = np.arange(100) / 100.0 * t_ast
    v_1_to_inf = 0.5 + w * np.tan(w / tau_m * t_blowup + np.arctan(1.0 / (2.0 * w)))

    v_minus_inf_to_0 = 1.0 - v_1_to_inf[::-1]
    return T, t_ast, v_0_to_1, v_1_to_inf, v_minus_inf_to_0


def plot_qif_infinite_threshold(T, t_ast, v_0_to_1, v_1_to_inf, v_minus_inf_to_0):
    t_period = np.arange(101) / 100.0 * T
    t_blowup = np.arange(100) / 100.0 * t_ast

    plt.figure(figsize=(10, 5))
    plt.plot(t_period, v_0_to_1, color='black', linewidth=3)
    plt.plot(T + t_blowup, v_1_to_inf, color='black', linewidth=1)
    plt.plot([T + t_ast, T + t_ast], [-1, 2], color='black', linestyle='dashed', linewidth=1)

    for ijk in range(1, 6):
        plt.plot(ijk * (T + 2 * t_ast) - t_ast + t_blowup, v_minus_inf_to_0, color='black', linewidth=1)
        plt.plot(ijk * (T + 2 * t_ast) + t_period, v_0_to_1, color='black', linewidth=3)
        plt.plot(ijk * (T + 2 * t_ast) + T + t_blowup, v_1_to_inf, color='black', linewidth=1)
        plt.plot([ijk * (T + 2 * t_ast) + T + t_ast, ijk * (T + 2 * t_ast) + T + t_ast],
                 [-1, 2], color='black', linestyle='dashed', linewidth=1)

    plt.xlabel('$t$')
    plt.ylabel('$v$')
    plt.ylim((-1, 2))
    plt.xlim((0, 150))
    plt.show()

In [ ]:
plot_qif_infinite_threshold(*simulate_qif_infinite_threshold())

## The QIF Neuron, Numerically Integrated

Heun (RK2) integration with a threshold-and-reset rule, matching the
book's incrementally-plotted trace (solid while subthreshold, dashed
through the reset).

In [ ]:
def simulate_qif_voltage_trace(tau_m=2.0, i=0.15, t_final=150.0, dt=0.01):
    dt05 = dt / 2
    num_steps = int(t_final / dt)
    t = np.linspace(0.0, t_final, num_steps)
    v = np.zeros_like(t)
    reset = np.zeros(num_steps, dtype=bool)
    for k in range(num_steps - 1):
        v_inc = -v[k] / tau_m * (1.0 - v[k]) + i
        v_tmp = v[k] + dt05 * v_inc
        v_inc = -v_tmp / tau_m * (1 - v_tmp) + i
        v_new = v[k] + dt * v_inc
        if v_new <= 1.0:
            v[k + 1] = v_new
        else:
            v[k + 1] = 0.0
            reset[k + 1] = True
    return t, v, reset


def plot_qif_voltage_trace(t, v, reset):
    plt.figure(figsize=(10, 5))
    for k in range(len(t) - 1):
        style = 'dashed' if reset[k + 1] else 'solid'
        plt.plot([t[k], t[k + 1]], [v[k], v[k + 1]], color='black', linestyle=style)
    plt.xlabel('$t$')
    plt.ylabel('$v$')
    plt.ylim((0.0, 2.0))
    plt.xlim((0.0, 150.0))
    plt.show()

In [ ]:
plot_qif_voltage_trace(*simulate_qif_voltage_trace())

In [ ]:
interact(lambda i=0.15: plot_qif_voltage_trace(*simulate_qif_voltage_trace(i=i)),
         i=(0.13, 0.5, 0.01));

## The Theta Neuron

The exact circle-variable rewriting of the QIF neuron; plotted as
$1-\cos\theta$ so its firing events look like voltage spikes.

In [ ]:
def simulate_theta_firing(tau_m=0.5, i=0.505, t_final=150.0, dt=0.001):
    dt05 = dt / 2.0
    m_steps = int(t_final / dt)
    t = np.linspace(0, t_final, m_steps + 1)
    theta = np.zeros(m_steps + 1)
    for k in range(m_steps):
        theta_inc = -np.cos(theta[k]) / tau_m + 2.0 * i * (1.0 + np.cos(theta[k]))
        theta_tmp = theta[k] + dt05 * theta_inc
        theta_inc = -np.cos(theta_tmp) / tau_m + 2.0 * i * (1.0 + np.cos(theta_tmp))
        theta[k + 1] = theta[k] + dt * theta_inc
    return t, theta


def plot_theta_firing(t, theta):
    plt.figure(figsize=(10, 5))
    plt.plot(t, 1.0 - np.cos(theta))
    plt.xlabel(r'$t$')
    plt.ylabel(r'$1-\cos(\theta)$')
    plt.ylim((0.0, 2.0))
    plt.xlim((0.0, 150.0))
    plt.xticks(np.arange(0, 151.0, step=50))
    plt.yticks(np.arange(0, 2.1, step=.50))
    plt.grid()
    plt.show()

In [ ]:
plot_theta_firing(*simulate_theta_firing())

In [ ]:
interact(lambda i=0.505: plot_theta_firing(*simulate_theta_firing(i=i)),
         i=(0.25, 1.0, 0.01));

## Fixed Points on the Theta-Neuron Circle

A schematic (no simulation) of the three qualitative regimes as $I$ crosses
$1/(4\tau_m)$: two fixed points (stable + unstable), a single half-stable
fixed point, and no fixed points (sustained firing).

In [ ]:
def plot_three_circles():
    theta = np.linspace(0, 2 * np.pi, 101)
    x = np.cos(theta)
    y = np.sin(theta)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4.5))

    ax = axes[0]
    ax.plot(x, y, color='k', linewidth=2)
    eps = 0.15
    theta0 = -0.4 * np.pi
    x0, y0 = np.cos(theta0), np.sin(theta0)
    ax.fill(x0 + eps * x, y0 + eps * y, 'k')
    y0 = -y0
    ax.fill(x0 + eps * x, y0 + eps * y, 'w', linewidth=1, edgecolor='k')
    for theta0 in [0.4, -0.4]:
        x0, y0 = np.cos(theta0), np.sin(theta0)
        v = np.array([np.sin(theta0 + 0.1), -np.cos(theta0 + 0.1)])
        draw_arrow(ax, [-1.5, 1.5], [-1.5, 1.5], x0, y0, v)
    for theta0 in [-2 / 3 * np.pi, 2 / 3 * np.pi, np.pi]:
        x0, y0 = np.cos(theta0), np.sin(theta0)
        v = -np.array([np.sin(theta0 - 0.1), -np.cos(theta0 - 0.1)])
        draw_arrow(ax, [-1.5, 1.5], [-1.5, 1.5], x0, y0, v)
    ax.text(-1.0, 1.6, r'$I < 1/(4\tau_m)$', fontsize=14)
    ax.axis([-1.5, 1.5, -1.5, 1.5])
    ax.set_aspect('equal')
    ax.axis('off')

    ax = axes[1]
    ax.plot(x, y, color='k', linewidth=2)
    theta_half = np.linspace(0, np.pi, 101)
    ax.fill(np.concatenate(([1 - eps, 1 + eps], 1 + eps * np.cos(theta_half))),
            np.concatenate(([0, 0], -eps * np.sin(theta_half))), 'k')
    ax.fill(np.concatenate(([1 - eps, 1 + eps], 1 + eps * np.cos(theta_half))),
            np.concatenate(([0, 0], eps * np.sin(theta_half))), 'w', linewidth=1, edgecolor='k')
    for theta0 in [-2 / 3 * np.pi, 1 / 3 * np.pi, -1 / 3 * np.pi, 2 / 3 * np.pi, np.pi]:
        x0, y0 = np.cos(theta0), np.sin(theta0)
        v = -np.array([np.sin(theta0 - 0.1), -np.cos(theta0 - 0.1)])
        draw_arrow(ax, [-1.5, 1.5], [-1.5, 1.5], x0, y0, v)
    ax.text(-1.0, 1.6, r'$I = 1/(4\tau_m)$', fontsize=14)
    ax.axis([-1.5, 1.5, -1.5, 1.5])
    ax.set_aspect('equal')
    ax.axis('off')

    ax = axes[2]
    ax.plot(x, y, color='k', linewidth=2)
    for theta0 in [-2 / 3 * np.pi, 1 / 3 * np.pi, -1 / 3 * np.pi, 2 / 3 * np.pi, np.pi, 0]:
        x0, y0 = np.cos(theta0), np.sin(theta0)
        v = -np.array([np.sin(theta0 - 0.1), -np.cos(theta0 - 0.1)])
        draw_arrow(ax, [-1.5, 1.5], [-1.5, 1.5], x0, y0, v)
    ax.text(-1.1, 1.6, r'$I > 1/(4\tau_m)$', fontsize=14)
    ax.axis([-1.5, 1.5, -1.5, 1.5])
    ax.set_aspect('equal')
    ax.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
plot_three_circles()